In [ ]:
# ============================================================
# Example.ipynb — fixed for your repo structure
# ------------------------------------------------------------
# Repo structure assumed:
# project-root/
#   ├─ src/BayesianVariableSelection/...
#   ├─ data/examples/X.csv
#   ├─ data/examples/Y.csv
#   ├─ data/raw/251205_ParisHotel_comment_score.csv
#   ├─ output/
#   └─ notebook/Example.ipynb   (this notebook)
# ============================================================

In [1]:
# ============================================================
# CELL 1) Project root + sys.path setup (run once per kernel)
# ============================================================
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    """Find project root by walking upward until we see both 'src' and 'data' folders."""
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "src").exists() and (p / "data").exists():
            return p
    # Fallback if detection fails (assumes notebook is project-root/notebook/)
    return start.parents[1]

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("SRC_DIR       =", SRC_DIR)

PROJECT_ROOT = /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection
SRC_DIR       = /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/src


In [2]:
# ============================================================
# CELL 2) Reload ALL project modules (run after code edits)
# ============================================================
import importlib

def reload_bvs_modules():
    """
    Reload BayesianVariableSelection modules without restarting the kernel.

    NOTE:
    - reload() requires module objects, so we import modules (not classes) first.
    - Order matters: reload lower-level deps (utils) before higher-level modules (bvs, label_bvs, etc.)
    """
    # ---- import modules (module objects needed for importlib.reload) ----
    import BayesianVariableSelection.utils as utils
    import BayesianVariableSelection.quality as quality
    import BayesianVariableSelection.bvs_pre as bvs_pre
    import BayesianVariableSelection.bvs as bvs
    import BayesianVariableSelection.label_bvs as label_bvs
    import BayesianVariableSelection.unstructured as unstructured
    import BayesianVariableSelection.word.frequency as word_frequency
    import BayesianVariableSelection.word.cluster as word_cluster

    modules_in_order = [
        utils,
        quality,
        word_frequency,
        word_cluster,
        bvs_pre,
        bvs,
        label_bvs,
        unstructured,
    ]

    for m in modules_in_order:
        importlib.reload(m)

    print("✔ Reloaded BayesianVariableSelection modules")

# Call once at the start (optional), then re-run this cell after edits
reload_bvs_modules()

✔ Reloaded BayesianVariableSelection modules


In [3]:
# ============================================================
# CELL 3) Imports (classes/functions you actually use)
# ============================================================
import pandas as pd

# Import classes AFTER reload so you get latest definitions
from BayesianVariableSelection.bvs import BayesianVariableSelection
from BayesianVariableSelection.label_bvs import LabelBVS
from BayesianVariableSelection.unstructured import UnstructureVariableSelection
from BayesianVariableSelection.word.frequency import WordFrequencyGenerator
from BayesianVariableSelection.word.cluster import WordClusterAnalyzer


In [4]:
# ============================================================
# CELL 4) Paths (single source of truth)
# ============================================================
DATA_EXAMPLES_DIR = PROJECT_ROOT / "data" / "examples"
DATA_RAW_DIR      = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR        = PROJECT_ROOT / "output"

X_PATH = DATA_EXAMPLES_DIR / "X.csv"
Y_PATH = DATA_EXAMPLES_DIR / "Y.csv"
RAW_PARIS_PATH = DATA_RAW_DIR / "251205_ParisHotel_comment_score.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("X_PATH        =", X_PATH)
print("Y_PATH        =", Y_PATH)
print("RAW_PARIS_PATH=", RAW_PARIS_PATH)
print("OUTPUT_DIR    =", OUTPUT_DIR)

X_PATH        = /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/data/examples/X.csv
Y_PATH        = /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/data/examples/Y.csv
RAW_PARIS_PATH= /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/data/raw/251205_ParisHotel_comment_score.csv
OUTPUT_DIR    = /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/output


In [8]:
# ============================================================
# CELL 5) Load data
# ============================================================
# Example data (structured)
X = pd.read_csv(X_PATH)
Y = pd.read_csv(Y_PATH)

# Make y 1D if it is a single-column label file
y = Y.iloc[:, 0] if isinstance(Y, pd.DataFrame) and Y.shape[1] == 1 else Y

print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("y type:", type(y), "/ shape:", getattr(y, "shape", None))

# Raw text data (unstructured)
raw_df = pd.read_csv(RAW_PARIS_PATH)
#print("raw_df shape:", raw_df.shape)
#raw_df.head(3)

X shape: (1000, 100)
Y shape: (1000, 1)
y type: <class 'pandas.core.series.Series'> / shape: (1000,)


In [31]:
# ============================================================
# CELL 6) Validate key fields + build "content" (title + text)
# - NO artificial tokens like "[REVIEW]"
# - NO newlines (one-line content)
# - Removes duplicated title prefix from text
# - Always prints nullability / effective-null stats
# ============================================================

import re
import numpy as np
import pandas as pd

# -----------------------------
# 6A) Required columns
# -----------------------------
REQ_COLS = ["rating", "locationId", "title", "text"]
missing = [c for c in REQ_COLS if c not in raw_df.columns]
if missing:
    raise KeyError(f"Missing required columns in raw_df: {missing}")

# -----------------------------
# 6B) Normalize dtypes (safe)
# -----------------------------
raw_df["rating"] = pd.to_numeric(raw_df["rating"], errors="coerce")
raw_df["locationId"] = raw_df["locationId"].astype("string")
raw_df["title"] = raw_df["title"].astype("string")
raw_df["text"] = raw_df["text"].astype("string")

# -----------------------------
# 6C) Helpers: "effective null" for text
# -----------------------------
def eff_empty(s: pd.Series) -> pd.Series:
    return s.isna() | s.fillna("").astype(str).str.strip().eq("")

def null_report(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = []
    n = len(df)
    for c in cols:
        is_null = df[c].isna()
        eff_null = eff_empty(df[c]) if c in ["title", "text", "content"] else is_null
        out.append({
            "col": c,
            "rows": n,
            "null_count": int(is_null.sum()),
            "null_ratio": float(is_null.mean()) if n else 0.0,
            "effective_null_count": int(eff_null.sum()),
            "effective_null_ratio": float(eff_null.mean()) if n else 0.0,
        })
    return pd.DataFrame(out).sort_values("effective_null_ratio", ascending=False)

display(null_report(raw_df, ["rating", "locationId", "title", "text"]))

# -----------------------------
# 6D) Smart merge(title, text) WITHOUT artificial tokens
# -----------------------------
def _norm_for_match(x: str) -> str:
    # normalize for rough matching: remove punctuation/spaces, lowercase
    x = (x or "").lower().strip()
    return re.sub(r"[^a-z0-9]+", "", x)

def _dedupe_title_prefix(title: str, text: str) -> str:
    """
    Remove title-like prefix from the start of text, if present.
    This prevents: "Madame Reve ALL THE WAY!!!" appearing twice.
    """
    title_s = (title or "").strip()
    text_s = (text or "").strip()
    if not title_s or not text_s:
        return text_s

    nt = _norm_for_match(title_s)
    nx = _norm_for_match(text_s)

    # Avoid dedupe when title too short (risk of false matches)
    if len(nt) < 12:
        return text_s

    # If text starts with title (normalized), cut it out
    if nx.startswith(nt):
        cut = len(title_s)
        return text_s[cut:].lstrip(" \n\r\t-–—:|")

    # If title occurs extremely early (e.g., "Title: Title body")
    idx = text_s.lower().find(title_s.lower())
    if 0 <= idx <= 5:
        return text_s[idx + len(title_s):].lstrip(" \n\r\t-–—:|")

    return text_s

def build_content(title: str, text: str) -> str:
    title_s = (title or "").strip()
    text_s  = (text or "").strip()

    if not title_s and not text_s:
        return ""
    if not title_s:
        return text_s
    if not text_s:
        return title_s

    body = _dedupe_title_prefix(title_s, text_s).strip()
    if not body:
        return title_s

    # One-line join, no newlines, no injected words
    # Using ". " is natural and doesn't introduce a systematic token like "REVIEW".
    return f"{title_s}. {body}"

raw_df["content"] = [
    build_content(t, x)
    for t, x in zip(raw_df["title"].fillna("").astype(str), raw_df["text"].fillna("").astype(str))
]

display(null_report(raw_df, ["content"]))

# -----------------------------
# 6E) Usable rows rule (your core requirements)
# -----------------------------
rating_ok = raw_df["rating"].between(1, 5, inclusive="both")
location_ok = raw_df["locationId"].notna() & raw_df["locationId"].astype(str).str.strip().ne("")
content_ok = ~eff_empty(raw_df["content"])

raw_df["is_usable"] = rating_ok & location_ok & content_ok

print(f"Usable rows: {raw_df['is_usable'].sum():,}/{len(raw_df):,} ({raw_df['is_usable'].mean():.2%})")

# Quick drop reasons
drop_reason = pd.Series(np.where(~rating_ok, "bad_or_missing_rating",
                        np.where(~location_ok, "missing_locationId",
                        np.where(~content_ok, "empty_content", "ok"))))
display(drop_reason.value_counts())

# Filtered dataset for downstream
df = raw_df.loc[raw_df["is_usable"]].copy()

# Optional: whitespace normalization (safe for tokenizers)
df["content_clean"] = (
    df["content"].astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Choose text column for your unstructured pipeline
TEXT_COL = "content_clean"  # recommended (same meaning, less noise)
# TEXT_COL = "content"      # if you want to preserve original spacing/punctuation more

# Sanity preview: confirm no duplicated title + no injected tokens
display(df[["title", "text", "content"]].head(5))
print("TEXT_COL =", TEXT_COL)

,col,rows,null_count,null_ratio,effective_null_count,effective_null_ratio
2,title,103885,213,0.00205,213,0.00205
0,rating,103885,0,0.00000,0,0.00000
1,locationId,103885,0,0.00000,0,0.00000
3,text,103885,0,0.00000,0,0.00000


,col,rows,null_count,null_ratio,effective_null_count,effective_null_ratio
0,content,103885,0,0.0,0,0.0


Usable rows: 103,885/103,885 (100.00%)


ok    103885
Name: count, dtype: int64

,title,text,content
0,Wonderful 5-star experience in the 1st. Perfe...,"This was our first visit, and it was a great e...",Wonderful 5-star experience in the 1st. Perfe...
1,Madame Reve ALL THE WAY!!!,We decided to go to Paris for our honeymoon an...,Madame Reve ALL THE WAY!!!. We decided to go t...
2,Overpriced,My partner and I stayed in a Deluxe room with ...,Overpriced. My partner and I stayed in a Delux...
3,Shocking breach of privacy at a so-called five...,"I liked the location and the room itself, but ...",Shocking breach of privacy at a so-called five...
4,Paris,Good location !\nThe service in the room was i...,Paris. Good location !\nThe service in the roo...


TEXT_COL = content_clean


In [32]:
# ============================================================
# CELL 7) Unstructured pipeline (TEXT -> word frequency -> LabelBVS)
#
# Assumes from CELL 6:
#   - df            : cleaned/filtered DataFrame (usable rows only)
#   - TEXT_COL      : e.g., "content_clean"
# And you have:
#   - rating column : "rating"
#   - group column  : "locationId" (aggregator)
#
# What this cell does:
#   1) Build word-frequency matrix X_words and weight/labels y_words
#   2) Fit Bayesian variable selection (LabelBVS) on word features
#
# NOTE:
#   The Bayesian part can be computationally heavy (MCMC). I separate
#   "lightweight steps" vs "run when ready" steps clearly.
# ============================================================

from pathlib import Path

# -----------------------------
# 7A) Column definitions
# -----------------------------
REVIEW_COL = TEXT_COL         # e.g., "content_clean"
RATING_COL = "rating"
AGG_COL    = "locationId"     # aggregator -> produces doc-level rows

# -----------------------------
# 7B) Where to save word corpus list (optional)
# -----------------------------
WORD_CORPUS_XLSX = OUTPUT_DIR / "word_corpus.xlsx"  # you can rename freely


# ============================================================
# CELL 7.1) Generate word frequency matrix (lightweight)
# ============================================================
from BayesianVariableSelection.unstructured import UnstructureVariableSelection

# Create model that combines:
# - WordFrequencyGenerator (text -> frequency matrix)
# - LabelBVS initializer prep
#
# word_corpus_file:
#   - If None: uses stopwords removal + lemmatization (default behavior)
#   - If you pass an Excel corpus file: restricts/standardizes vocabulary
#
unstruct = UnstructureVariableSelection(
    data=df,                    # IMPORTANT: use filtered df from Cell 6
    review_col=REVIEW_COL,
    rating_col=RATING_COL,
    aggregate_col=AGG_COL,
    word_corpus_file=None,      # change to WORD_CORPUS_XLSX later if you want controlled vocab
    dist_type="correlate",      # Distance Type: correlate (Correlation; aggregate) / cooccur (Cooccurrence; review-level)
    track=True                  # set True if you want progress tracking/visual widgets
)

# Generate frequency matrix
# Options:
#   - word_count=N  : keep words that appear at least N times in a document (threshold)
#   - top_n=N       : keep top N words per document
#
# Choose ONE (do not set both). Start small if debugging.
X_words, y_words = unstruct.generate(top_n=300)

print("X_words shape:", X_words.shape)   # (num_docs, vocab_size)
print("y_words shape:", y_words.shape)   # (num_docs, 2) -> count, mean_rating
X_words.head(3)


# ============================================================
# CELL 7.2) Export word corpus (optional but recommended)
# ============================================================
# This exports the current vocabulary to Excel.
# Later you can edit it to control mapping, synonyms, and allowed tokens.
# Format expected by your code:
#   columns: ["valid_corpus", "change"]
#
# - valid_corpus: allowed token
# - change      : (optional) token replacement (synonym mapping)
#
try:
    unstruct.export_word_corpus(WORD_CORPUS_XLSX)
    print("Saved word corpus:", WORD_CORPUS_XLSX)
except Exception as e:
    print("[WARN] export_word_corpus failed (safe to ignore if you don't need it):", e)

300 of words are selected
X_words shape: (904, 300)
y_words shape: (904, 2)
Saved word corpus: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/output/word_corpus.xlsx


In [22]:
# pip install -U openai python-dotenv pandas openpyxl

import json
import math
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# ============================================================
# 0) Load env (.env at repo root)
# ============================================================
load_dotenv()
client = OpenAI()

# ============================================================
# 1) Input / Output paths
# ============================================================
# Must be defined before running this cell/script
# Example:
# WORD_CORPUS_XLSX = "./output/word_corpus.xlsx"
in_path = Path(WORD_CORPUS_XLSX)
out_path = in_path.with_name(in_path.stem + "_cleaned.xlsx")

# ============================================================
# 2) Load workbook
# ============================================================
df = pd.read_excel(in_path)

# ============================================================
# 3) Detect token column
# ============================================================
def detect_word_col(columns):
    cols = list(columns)
    low = {str(c).strip().lower(): c for c in cols}
    for k in ("word", "token", "vocab", "term", "original", "valid_corpus"):
        if k in low:
            return low[k]
    return cols[0]

word_col = detect_word_col(df.columns)

# ============================================================
# 4) Ensure lemma column exists (NO other new columns)
# ============================================================
if "lemma" not in df.columns:
    df["lemma"] = pd.NA

# If you want lemma immediately to the right of word_col:
cols = [c for c in df.columns if c != "lemma"]
cols.insert(cols.index(word_col) + 1, "lemma")
df = df.reindex(columns=cols)

# ============================================================
# 5) Normalize word cells + drop blank rows
# ============================================================
def norm_cell(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s.lower() in ("nan", "none"):
        return ""
    return s

df[word_col] = df[word_col].apply(norm_cell)
df = df[df[word_col].ne("")].copy()

words = df[word_col].unique().tolist()

# ============================================================
# 6) Deterministic noise filter (drop without API)
# ============================================================
def is_obviously_noise(w: str) -> bool:
    if not w or len(w) <= 1:
        return True
    if w.isdigit():
        return True
    if not any(ch.isalpha() for ch in w):
        return True
    wl = w.lower()
    if "http" in wl or "www." in wl or "@" in wl:
        return True
    if re.fullmatch(r"[\W_]+", w):
        return True
    return False

obvious_noise_set = {w for w in words if is_obviously_noise(w)}
words_for_api = [w for w in words if w not in obvious_noise_set]

# ============================================================
# 7) OpenAI review: keep + corrected_word (in-place) + lemma (optional)
# ============================================================
BATCH_SIZE = 150
MODEL = "gpt-4o-mini"

SYSTEM_INSTRUCTIONS = """
You are cleaning a vocabulary list for a text-based statistical model
that relies on word frequency, co-occurrence, and Bayesian variable selection.

For each input token, output a JSON object with:
- word: original string (exactly)
- keep: true/false (remove if inappropriate for modeling)
- corrected: corrected/normalized token (only if needed; else return the original)
- lemma: "" OR a base/root form WHEN morphological variants would otherwise
         split signal across related tokens.

IMPORTANT LEMMA RULE:
- If an adjective/adverb/verb is clearly derived from a common noun or verb root,
  provide the root as lemma.
  Examples:
    comfortable → comfort
    comforting → comfort
    cleanliness → clean
    cleaner → clean
    cleanest → clean
- Do NOT invent lemmas for proper nouns, hotel names, locations, or brands.
- If the word is already the root form, leave lemma empty.

Keep criteria:
- Content words carrying sentiment, quality, or topic signal.
- Domain-relevant terms in hospitality reviews.

Remove criteria:
- Stopwords, filler words, artifacts, punctuation fragments,
  encoding noise, URLs/emails/IDs, meaningless tokens.

Return STRICT JSONL (one JSON object per line).
No markdown. No commentary.

"""

def call_review_batch(batch_words):
    payload = {"words": batch_words}
    resp = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_INSTRUCTIONS,
        input=json.dumps(payload, ensure_ascii=False),
    )
    text = (resp.output_text or "").strip()
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except Exception:
            continue
    return rows

review_map = {}  # word -> {keep, corrected, lemma}

num_batches = math.ceil(len(words_for_api) / BATCH_SIZE) if words_for_api else 0
for i in range(num_batches):
    batch = words_for_api[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
    result_rows = call_review_batch(batch)
    for r in result_rows:
        w = str(r.get("word", "")).strip()
        if not w:
            continue
        review_map[w] = {
            "keep": bool(r.get("keep", True)),
            "corrected": (r.get("corrected") or w).strip(),
            "lemma": (r.get("lemma") or "").strip(),
        }

# ============================================================
# 8) Apply decisions:
# - drop rows if keep=False
# - overwrite word in-place if corrected differs
# - write lemma only if non-empty
# ============================================================
def apply_decision(w: str):
    # drop obvious noise
    if w in obvious_noise_set:
        return False, w, ""

    ann = review_map.get(w)
    if ann is None:
        # fallback: keep as-is, no lemma
        return True, w, ""

    keep = ann["keep"]
    corrected = ann["corrected"] or w
    lemma = ann["lemma"] or ""
    return keep, corrected, lemma

decisions = df[word_col].apply(apply_decision)

df["__keep"] = decisions.map(lambda t: t[0])
df[word_col] = decisions.map(lambda t: t[1])  # corrected in-place
df["lemma"] = decisions.map(lambda t: t[2] if t[2] else pd.NA)

before_n = len(df)
df = df[df["__keep"]].drop(columns=["__keep"]).copy()
after_n = len(df)

print(f"Dropped rows: {before_n - after_n:,} / {before_n:,}")
print(f"Remaining rows: {after_n:,}")
print(f"Token column detected: '{word_col}' | unique tokens: {len(words):,} | sent to API: {len(words_for_api):,}")

# ============================================================
# 9) Save
# ============================================================
df.to_excel(out_path, index=False)
print("Cleaned vocab saved to:", out_path)

Dropped rows: 102 / 300
Remaining rows: 198
Token column detected: 'corpus' | unique tokens: 300 | sent to API: 293
Cleaned vocab saved to: /Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/output/word_corpus_cleaned.xlsx


In [ ]:
# ============================================================
# CELL 7-3) (Important future code) Controlled vocabulary workflow
#
# Why:
# - You export the detected corpus once
# - Manually clean / merge synonyms / remove junk tokens
# - Re-run frequency generation using that controlled corpus file
#
# Steps:
#   1) Run Cell 7.2 once -> creates output/word_corpus.xlsx
#   2) Open and edit it:
#       - remove garbage tokens
#       - set "change" to normalize synonyms (e.g., "hotel" -> "hotel")
#   3) Rebuild unstructured model with word_corpus_file=WORD_CORPUS_XLSX
# ============================================================

# Rebuild unstructured model with controlled vocab
unstruct_vocab = UnstructureVariableSelection(
    data=df,
    review_col=REVIEW_COL,
    rating_col=RATING_COL,
    aggregate_col=AGG_COL,
    word_corpus_file=out_path,  # <-- now controlled
    dist_type="correlate",
    track=True
)

_ = unstruct_vocab.generate()  

# Then run Bayesian steps as usual
unstruct_vocab.initialize(segment_no=2, iteration=3000, burnin=2000, cutoff=0.70, step=1)
unstruct_vocab.fit()
# unstruct_vocab.estimation()
# unstruct_vocab.finalize()
print("✔ Unstructured with controlled vocabulary finished.")


In [ ]:
unstruct_vocab.visualize_topic_seg(0, k = 3, cut = 0.87).update_layout(width=1200, height=600, xaxis_tickangle = 45)

In [ ]:
# ============================================================
# CELL Extra) Fit Bayesian selection on word features (RUN WHEN READY)
# ============================================================
# The Bayesian selection model is created inside `unstruct.generate()`
# via LabelBVS.__init__(X=..., y=mean_rating, weight=count, ...)
#
# So after generate(), `unstruct` *is also* a fitted LabelBVS-ready model.
#
# Next steps:
#   1) initialize(K segments, iterations, burnin, etc.)
#   2) estimation()  -> runs MCMC
#   3) finalize()    -> finalize posterior summaries
#
# ⚠️ This may take time depending on iteration/burnin.
# Start small (e.g., iteration=200, burnin=200) to confirm it runs.
# Then scale up.

# ---- START SMALL for debugging ----
SEGMENTS = 1       # aka segment_no (latent segments K)
ITER     = 1000    # MCMC iterations
BURNIN   = 2000    # burn-in draws / default is 2000
STEP     = 1       # save every STEP draws
CUTOFF   = 0.70    # Correlate: cutoff = 0.7 / Cooccur: cutoff = 0.3


unstruct.initialize(
    segment_no=SEGMENTS,
    iteration=ITER,
    burnin=BURNIN,
    cutoff=CUTOFF,
    step=STEP,
    hyperparameters=None
)

unstruct.estimation()
unstruct.finalize()

print("✔ Unstructured LabelBVS finished.")

KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL Extra) Post-fit utilities (safe to run after finalize)
# ============================================================

# (A) Export the X/y used in Bayesian regression
# - X includes intercept + selected words (LabelBVS overrides column names)
X_export, y_export = unstruct.export_data()
print("X_export shape:", X_export.shape)
print("y_export shape:", y_export.shape)

# (B) Find group members (word clusters) for a segment
# seg_no: which segment to inspect
# k     : number of groups inside the segment
# cut   : cut for co-occurrence / network thresholding
seg_no = 0
k = 3
cut = 0.6

group_members = unstruct.find_group_members(seg_no=seg_no, k=k, cut=cut)
group_members  # dict: group -> list of words


# (C) Optional visual tools (if you want them)
# unstruct.visualize_topic_seg(seg_no=0, k=3, cut=0.87)
# unstruct.visualize_heatmap(seg_no=0, k=3, cut=0.87)

X_export shape: (904, 301)
y_export shape: (904,)


ValueError: not enough values to unpack (expected 4, got 1)

In [ ]:
# ============================================================
# CELL Extra) Word clustering / dimensional reduction (optional, but powerful)
# ============================================================
from BayesianVariableSelection.word.cluster import WordClusterAnalyzer

# Requires: fitted unstruct model (after finalize), because it uses group membership
wca = WordClusterAnalyzer(
    model=unstruct,
    seg_no=0,          # which latent segment to analyze
    k=3,               # groups within segment
    mode="PCA",        # "PCA" or "CFA" (ConfirmatoryFactorAnalyzer) or "CAL" (quality calculator)
    cluster_label=None # or pass a dict to override automatic groups
)

wca.fit(cut=0.95)

# Transform original df into reduced dimensions (adds cluster-derived features)
# You can pass df (dataframe) or the model object
df_reduced = wca.transform(df.copy())

print("df_reduced shape:", df_reduced.shape)
df_reduced.head(3)

In [ ]:
# ============================================================
# CELL 8) Structured pipeline (X.csv / Y.csv) using BayesianVariableSelection
#
# This is separate from text-based pipeline.
# Use when you want BVS on existing structured features (X) -> labels (y).
#
# Assumes from CELL 5:
#   - X, y are loaded (X is DataFrame, y is Series or array-like)
# ============================================================

from BayesianVariableSelection.bvs import BayesianVariableSelection

# If y is a DataFrame with one column, convert
if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
    y_struct = y.iloc[:, 0]
else:
    y_struct = y

# Optional: weight (if you have observation weights)
weight = None

bvs_struct = BayesianVariableSelection(
    X=X,
    y=y_struct,
    weight=weight,
    dist_type="cooccur",  # keep default unless you know otherwise
    track=False
)

# Correlate: cutoff = 0.7 / Cooccur: cutoff = 0.3
# Start small to verify it runs
bvs_struct.initialize(
    segment_no=1,
    iteration=1000,
    burnin=2000,
    cutoff=0.70,
    step=1,
    random_seed=42,
    hyperparameters=None
)

bvs_struct.estimation()
bvs_struct.finalize()

print("✔ Structured BVS finished.")

# Explore cluster/segment structure
members = bvs_struct.find_group_members(seg_no=0, k=2, cut=0.95)
members


# The code below is an example created by 경돈.

### It contains:
### 1. Basic Bayesian variable selection
### 2. Labeled Bayesian variable selection
### 3. Unstructured variable selection that runs both (1) and (2)

In [ ]:
selector = BayesianVariableSelection(
    X.fillna(0).to_numpy(),
    y.to_numpy().reshape(-1),
    track = True,
)

In [ ]:
# Model Initialize
selector.initialize(segment_no = 3, iteration = 3000, burnin = 2000, cutoff = 0.03, step = 1)

In [ ]:
selector.fit()

  0%|          | 0/5000 [00:00<?, ?it/s]

In [ ]:
selector.visualize_topic_seg(0)

In [ ]:
# Model Setup
from BayesianVariableSelection import LabelBVS

selector = LabelBVS(
    X.fillna(0), # X: word frequency matrix
    y['x'], # y: rating 
    dist_type = 'cooccur',
    track = True,
)

In [ ]:
# Model Initialize
selector.initialize(segment_no = 3, iteration = 3000, burnin = 2000, cutoff = 0.03, step = 1)

In [ ]:
selector.fit()

  0%|          | 0/5000 [00:00<?, ?it/s]

In [ ]:
selector.visualize_topic_seg(0)

In [ ]:
data = pd.read_csv('./data/251205_ParisHotel_comment_score.csv')

In [ ]:
data['content'] = data['title'].fillna('').str.cat(data['text'], sep=' ')

In [ ]:
# Review-level data
data[['id', 'locationId', 'content', 'rating']]

,id,locationId,content,rating
0,1009363551,23688737.0,Wonderful 5-star experience in the 1st. Perfe...,5.0
1,1041040518,23688737.0,Madame Reve ALL THE WAY!!! We decided to go to...,5.0
2,1040950353,23688737.0,Overpriced My partner and I stayed in a Deluxe...,3.0
3,1040312147,23688737.0,Shocking breach of privacy at a so-called five...,2.0
4,1040284170,23688737.0,Paris Good location !\nThe service in the room...,4.0
...,...,...,...,...
103880,834074624,617625.0,Great Boutique style hotel Great location clos...,5.0
103881,833748659,617625.0,Amazing property and amazing room We were so h...,5.0
103882,833729338,617625.0,We love this hotel. Me and my daughter have st...,5.0
103883,833478344,617625.0,Fantastic 3 nights at Grand Hotel du Palais Ro...,5.0


In [ ]:
data['locationId_month'] = data['locationId'].astype(str) + "_" + data['publishedDate'].astype('datetime64[ns]').dt.to_period('M').astype(str)

In [ ]:
from BayesianVariableSelection.UnstructureVariableSelection import UnstructureVariableSelection
selector = UnstructureVariableSelection(
    data = data,
    review_col = 'content',
    rating_col = 'rating',
    aggregate_col = 'locationId',
    word_corpus_file = None,
    dist_type = 'correlate', # Distance Type: correlate (Correlation; aggregate) / cooccur (Cooccurrence; review-level)
    track = True
)

In [ ]:
_ = selector.generate(top_n = 300)

300 of words are selected


In [ ]:

selector.export_word_corpus('./260106_word_corpus.xlsx')

In [ ]:
from BayesianVariableSelection.UnstructureVariableSelection import UnstructureVariableSelection
selector = UnstructureVariableSelection(
    data = data,
    review_col = 'content',
    rating_col = 'rating',
    aggregate_col = 'locationId',
    word_corpus_file = './260106_word_corpus.xlsx',
    dist_type = 'correlate', # Distance Type: correlate (Correlation; aggregate) / cooccur (Cooccurrence; review-level)
    track = True
)

In [ ]:
_ = selector.generate()

288 of words are selected


In [ ]:
# Correlate: cutoff = 0.7 / Cooccur: cutoff = 0.3
selector.initialize(segment_no = 3, iteration = 300, burnin = 300, cutoff = 0.7, step = 1)

In [ ]:
selector.fit()

  0%|          | 0/600 [00:00<?, ?it/s]

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning:

invalid value encountered in divide

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning:

invalid value encountered in divide

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning:

Degrees of freedom <= 0 for slice

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning:

divide by zero encountered in divide

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:2888: 

In [ ]:
selector.update(500)

  0%|          | 0/500 [00:00<?, ?it/s]

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning:

Degrees of freedom <= 0 for slice

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning:

divide by zero encountered in divide

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning:

invalid value encountered in multiply

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning:

invalid value encountered in divide

/Users/jinseunghwan/Documents/development/Data Processing/group-variable-selection/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024

In [ ]:
selector.visualize_topic_seg(2, k = 3, cut = 0.9).update_layout(width=1200, height=600, xaxis_tickangle = 45)

In [ ]:
np.unique(selector.find_membership(), return_counts=True)

(array([0, 1, 2]), array([ 30, 331, 543]))

In [ ]:
selector.find_appropriate_document('appreciate')

,locationId,rating,content
0,23688737.0,5.0,Wonderful 5-star experience in the 1st. Perfe...
1,23688737.0,5.0,Madame Reve ALL THE WAY!!! We decided to go to...
2,23688737.0,3.0,Overpriced My partner and I stayed in a Deluxe...
3,23688737.0,2.0,Shocking breach of privacy at a so-called five...
4,23688737.0,4.0,Paris Good location !\nThe service in the room...
...,...,...,...
103880,617625.0,5.0,Great Boutique style hotel Great location clos...
103881,617625.0,5.0,Amazing property and amazing room We were so h...
103882,617625.0,5.0,We love this hotel. Me and my daughter have st...
103883,617625.0,5.0,Fantastic 3 nights at Grand Hotel du Palais Ro...


In [ ]:
selector.diagnose_model()

,variable,parameter,std_err,t_stat,p_value,CI_low,CI_upp
3,absolutely,-0.5755,0.202,-2.856,0.004,-0.971,-0.180
4,access,0.4836,0.241,2.007,0.045,0.010,0.957
8,although,0.6288,0.240,2.625,0.009,0.158,1.099
9,always,-0.4031,0.151,-2.667,0.008,-0.700,-0.106
11,amazing,0.8505,0.298,2.851,0.005,0.265,1.436
...,...,...,...,...,...,...,...
272,value,0.5800,0.169,3.442,0.001,0.249,0.911
275,wait,0.4978,0.226,2.207,0.028,0.055,0.941
282,welcome,0.3281,0.112,2.921,0.004,0.107,0.549
288,work,-0.3908,0.123,-3.181,0.002,-0.632,-0.150


In [ ]:
selector.find_correlation_by_seg(0)

,index,select_ratio,coef|z=1,coef
corpus,,,,
absolutely,1,0.990000,-0.622800,-0.616572
arrival,15,1.000000,-0.949426,-0.949426
bad,25,1.000000,-0.623945,-0.623945
bathroom,28,1.000000,-0.440682,-0.440682
beautiful,29,1.000000,0.580114,0.580114
bed,30,1.000000,-0.340971,-0.340971
call,41,1.000000,-0.614966,-0.614966
comfort,55,1.000000,0.362199,0.362199
customer,62,1.000000,-0.801028,-0.801028


In [ ]:
freq = selector.frequency_matrix.copy()
freq['membership'] = selector.find_membership()

array([2, 1, 0, 2, 2, 2, 1, 2, 2, 1, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2, 1, 2,
       1, 2, 2, 2, 2, 1, 2, 2, 2, 1, 1, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2,
       2, 1, 2, 2, 1, 2, 2, 2, 1, 2, 1, 2, 2, 1, 2, 1, 2, 2, 1, 2, 1, 2,
       2, 2, 1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 2, 2, 2, 2, 0, 2, 1, 2, 1, 1,
       2, 1, 2, 0, 2, 1, 1, 2, 1, 1, 2, 1, 2, 2, 1, 2, 2, 2, 2, 1, 2, 2,
       2, 2, 1, 2, 2, 2, 1, 2, 1, 1, 2, 2, 1, 2, 1, 1, 1, 2, 2, 1, 2, 1,
       2, 2, 2, 1, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1, 2, 1, 2, 1, 1, 1,
       1, 2, 2, 2, 2, 2, 2, 0, 1, 2, 2, 1, 2, 2, 2, 2, 2, 1, 0, 2, 1, 1,
       2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 0, 2, 1, 1, 2, 2, 1,
       2, 2, 2, 2, 1, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1,
       2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 2, 2, 1, 2, 1, 2, 2, 2, 2, 2,
       1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2,
       1, 2, 2, 2, 2, 2, 2, 1, 1, 2, 1, 1, 0, 2, 2, 1, 2, 2, 1, 2, 1, 2,
       2, 2, 2, 1, 2, 2, 1, 1, 2, 1, 2, 2, 2, 1, 2,